In [1]:
# Make sure the local settings ntuple path points to the unfiltered ntuples and adjust data_loading.py

import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
sys.path.append("../../../")
import data_loading as dl
from importlib import reload
reload(dl)

from microfit import run_plotter as rp
from microfit import histogram as hist

from microfit import variable_definitions as vdef
from microfit import selections

In [2]:
resp = np.array([[0.13130073, 0.01889182],
                 [0.01249219, 0.09767357]])
# writing these to a file
resp_str = "{"
for i in range(resp.shape[0]):
    for j in range(resp.shape[1]):
        resp_str += f"{resp[i][j]},"
resp_str = resp_str[:-1] # to remove the last comma
resp_str += "};"
print(resp_str)

{0.13130073,0.01889182,0.01249219,0.09767357};


In [2]:
keep_vars = [
    "Signal_1e1p", "mc_signal_1e1p", "nu_pdg", "TrueElecIdx", "TrueLeadProtonIdx", "InFV", "HasNoMesons", "TrueNElec", "TrueNProt", 
    "TrueDeltaPT", "TrueDeltaAlphaT", "TruePN", "TrueAlpha3D",
    "TrueLeadProtonKE", "TrueLeadProtonModMom", "TrueLeadProtonE", "TrueLeadProtonMomX", "TrueLeadProtonMomY", "TrueLeadProtonMomZ",
    "TrueElecKE", "TrueElecModMom", "TrueElecE", "TrueElecMomX", "TrueElecMomY", "TrueElecMomZ",

    "Sel_1e1p", "sel_1e1p_w_cuts", "RecoElectronCandidateIdx", "RecoLeadProtonCandidateIdx", "InFV_reco",
    "RecoElecPassMomCut", "RecoLeadProtonPassMomCut", "n_reco_tracks", "n_reco_showers",
    "RecoDeltaPT", "RecoDeltaAlphaT", "RecoPN", "RecoAlpha3D", "RecoECal", "Reco_mag_q", "RecoPL",
    "RecoLeadProtonKE", "RecoLeadProtonModMom", "RecoLeadProtonMomX", "RecoLeadProtonMomY", "RecoLeadProtonMomZ",
    "RecoElecE", "RecoElecModMom", "RecoElecMomX", "RecoElecMomY", "RecoElecMomZ",

    "RecoLeadProton_trk_len", "RecoLeadProton_trk_trunk_dEdx_y", "RecoLeadProton_dEdx_y_per_trklen",
    "RecoLeadProtonCandidate_trk_pid", "RecoElectronCandidate_shr_pid", "RecoElectron_conversion_dist",

    "nproton", "npion", "npi0", "nelec", "nmuon", "isVtxInFiducial",

    "nslice", "selected", "shr_energy_tot_cali", "_opfilter_pe_beam", "_opfilter_pe_veto", "bnbdata", "extdata",
    "CosmicIPAll3D", "hits_ratio", "shrmoliereavg", "subcluster", "trkfit", "trkshrhitdist2", "tksh_distance",
    "shr_tkfit_nhits_tot", "shr_tkfit_dedx_max", "tksh_angle", "shr_trk_len", "reco_e",
    "trkpid", "trk_len", "n_showers_contained", "protonenergy_corr", "n_tracks_contained",
    "pi0_radlen1", "pi0_radlen2", "pi0_score", "nonpi0_score", "bkg_score",

    # "InFV_1muNp", "TrueMuonIdx_1muNp", "TrueLeadProtonIdx_1muNp", "TrueNProt_1muNp", "TrueFSPions_1muNp", "Signal_1mu1p", 
    # "TrueDeltaPT_1mu1p", "TrueDeltaAlphaT_1mu1p", "TruePN_1mu1p", "TrueAlpha3D_1mu1p",
    # "TrueLeadProtonE_1muNp", "TrueLeadProtonMomX_1muNp", "TrueLeadProtonMomY_1muNp", "TrueLeadProtonMomZ_1muNp",
    # "TrueMuonE_1muNp", "TrueMuonMomX_1muNp", "TrueMuonMomY_1muNp", "TrueMuonMomZ_1muNp",

    # "sel_CC1p0pi", "InFV_reco_1muNp", "MuonCandidateIdx_1muNp", "LeadProtonIdx_1muNp", "LeadProtonPassMomentumCut_1muNp", "PFPStartsInPCV_1muNp", "PassTopoScoreCut_1muNp",
    # "PassNuMuCCSelection_1muNp", "NoRecoShowers_1muNp", "MuonContained_1muNp", "PassMuonMomentumCut_1muNp", "PassMuonQualCut_1muNp",
    # "LeadProtonPassMomentumCut_1muNp", "NProtons_1muNp",
    # "RecoDeltaPT_1mu1p", "RecoDeltaAlphaT_1mu1p", "RecoPN_1mu1p", "RecoAlpha3D_1mu1p", "RecoECal_1mu1p", "RecoPL_1mu1p",
    # "RecoLeadProtonE_1muNp", "RecoLeadProtonMomentum_1muNp", "RecoLeadProtonMomX_1muNp", "RecoLeadProtonMomY_1muNp", "RecoLeadProtonMomZ_1muNp", 
    # "RecoMuonE_1muNp", "RecoMuonMomentum_1muNp", "RecoMuonMomX_1muNp", "RecoMuonMomY_1muNp", "RecoMuonMomZ_1muNp",
]

In [3]:
RUN = ["3"]
#RUN = ["1","2","3_nocrt","3_crt","4a","4b","4c","4d","5"] # use this if using CRT
#RUN = ["1","2","3","4a","4b","4c","4d","5","1A_OT","1B_OT"] # for detvars with bnb or for closure test
#RUN = ["1","2","3","4a","4c","5"] # for nuwro_fd, no run 4b and 4d available
blinded = True
#data="nuwro_fd"
data="bnb"
ingredients = True

rundata, mc_weights, data_pot = dl.load_runs(
    RUN,
    data=data,
    loadpi0variables=False,
    loadshowervariables=True,
    loadrecoveryvars=False,
    loadsystematics=True,
    numupresel=False,
    loadnumuvariables=False,
    use_bdt=True,
    load_lee=False,
    load_numu_tki=False,
    load_nue_tki=True,
    keep_columns=keep_vars,
    blinded=blinded,
    load_crt_vars=False,
    enable_cache=True,
)

print('Loaded data')

run_combo = "Run"
for run in RUN:
    run_combo += run

Loading run 3
Updating keep_columns with truth-filtering variables: {'nu_pdg', 'ccnc'}
Loaded data


In [4]:
ingredient_vars = {
    'Proton Kinetic Energy [GeV]': {'reco': 'RecoLeadProtonKE', 'truth': 'TrueLeadProtonKE', 'nbins': 10, 'bounds': (0, 1), 'range': [[0, 1],[0, 1]]},
    'Proton Momentum [GeV/c]': {'reco': 'RecoLeadProtonModMom', 'truth': 'TrueLeadProtonModMom', 'nbins': 20, 'bounds': (0, 1.5), 'range': [[0, 1.5],[0, 1.5]]},
    'Proton X Momentum [GeV/c]': {'reco': 'RecoLeadProtonMomX', 'truth': 'TrueLeadProtonMomX', 'nbins': 20, 'bounds': (-1, 1), 'range': [[-1, 1],[-1, 1]]},
    'Proton Y Momentum [GeV/c]': {'reco': 'RecoLeadProtonMomY', 'truth': 'TrueLeadProtonMomY', 'nbins': 20, 'bounds': (-1.5, 1.5), 'range': [[-1.5, 1.5],[-1.5, 1.5]]},
    'Proton Z Momentum [GeV/c]': {'reco': 'RecoLeadProtonMomZ', 'truth': 'TrueLeadProtonMomZ', 'nbins': 20, 'bounds': (-1, 1.5),'range': [[-1, 1.5],[-1, 1.5]]},
    'Electron Energy [GeV]': {'reco': 'RecoElecE', 'truth': 'TrueElecE', 'nbins': 16, 'bounds': (0, 4), 'range': [[0, 4],[0, 4]]},
    'Electron Momentum [GeV/c]': {'reco': 'RecoElecModMom', 'truth': 'TrueElecModMom', 'nbins': 20, 'bounds': (0, 5), 'range': [[0, 5],[0, 5]]},
    'Electron X Momentum [GeV/c]': {'reco': 'RecoElecMomX', 'truth': 'TrueElecMomX', 'nbins': 12, 'bounds': (-1.5, 1.5), 'range': [[-1.5, 1.5],[-1.5, 1.5]]},
    'Electron Y Momentum [GeV/c]': {'reco': 'RecoElecMomY', 'truth': 'TrueElecMomY', 'nbins': 12, 'bounds': (-1.5, 1.5), 'range': [[-1.5, 1.5],[-1.5, 1.5]]},
    'Electron Z Momentum [GeV/c]': {'reco': 'RecoElecMomY', 'truth': 'TrueElecMomZ', 'nbins': 12, 'bounds': (-1, 5), 'range': [[-1, 5],[-1, 5]]},
}

variables = {
# #    '': {'reco': , 'truth': , 'nbins': , 'bounds': },
    '$\\delta p_T$ [GeV/c]': {'reco': 'RecoDeltaPT', 'truth': 'TrueDeltaPT', 'nbins': 20, 'bounds': (0, 2), 'range': [[0,1.7],[0,1.7]], 'bin_edges': [[0,0.3,1.7],[0,0.3,1.7]], 'bin_edges_1d': [0,0.3,1.7]},
    # '$\\delta \\alpha_T$ [degrees]': {'reco': 'RecoDeltaAlphaT', 'truth': 'TrueDeltaAlphaT', 'nbins': 20, 'bounds': (0, 180), 'range': [[0,180],[0,180]], 'bin_edges': [[0,80,180],[0,80,180]], 'bin_edges_1d': [0,80,180]},
    # '$p_n$ [GeV/c]': {'reco': 'RecoPN', 'truth': 'TruePN', 'nbins': 10, 'bounds': (0, 2), 'range': [[0,1.7],[0,1.7]], 'bin_edges': [[0,0.3,1.7],[0,0.3,1.7]], 'bin_edges_1d': [0,0.3,1.7]},
    # '$\\alpha_{3D}$ [degrees]': {'reco': 'RecoAlpha3D', 'truth': 'TrueAlpha3D', 'nbins': 10, 'bounds': (0, 180), 'range': [[0,180],[0,180]], 'bin_edges': [[0,90,180],[0,90,180]], 'bin_edges_1d': [0,90,180]},
}

In [5]:
selection = "OnePBDT"
preselection = "OneP_new"
        
# Calculating the response matrix:

from microfit import selections as sel
query = f"{sel.preselection_categories[preselection]['query']} and {sel.selection_categories[selection]['query']}"

all_mc = pd.concat([df for k, df in rundata.items() if k!='data' or k!='ext'])
is_sig = all_mc['category_1e1p'] == 12
all_sig = all_mc.loc[is_sig]
sel_sig = all_sig.query(query, engine='python')

: 

In [ ]:
if ingredients:
    for k, var in ingredient_vars.items():
        truth = ingredient_vars[k]['truth']
        reco = ingredient_vars[k]['reco']
        bounds = ingredient_vars[k]['bounds']
        range = ingredient_vars[k]['range']
        bin_edges = ingredient_vars[k]['nbins']
        bin_edges_1d = ingredient_vars[k]['nbins']
        
        truth_hist, truth_edges = np.histogram(all_sig[truth], bins=bin_edges_1d, weights=all_sig["weights"], range=bounds)
        H, xedges, yedges = np.histogram2d(sel_sig[truth], sel_sig[reco], bins=bin_edges, weights=sel_sig["weights"], range=range)
        
        resp = H.T / truth_hist
        
        X, Y = np.meshgrid(xedges,yedges)
        plt.pcolormesh(X, Y, resp, shading='flat') #, norm=LogNorm())
        plt.colorbar()
        plt.xlabel(f'True {k}')
        plt.ylabel(f'Reco {k}')
        
        plt.title(f"Response Matrix")
        label = reco.lstrip("Reco")
        plt.savefig(f'analysis_plots/unfolding_inputs/response_matrix_{data}_{run_combo}_{label}.pdf', bbox_inches='tight')
        plt.savefig(f'analysis_plots/unfolding_inputs/response_matrix_{data}_{run_combo}_{label}.png', bbox_inches='tight')
        plt.show()
        plt.clf()

In [ ]:
with open(f'unfolding_inputs_{data}_{run_combo}.txt', 'a') as f:
    f.write("\nResponse Matrices:\n")

for k, var in variables.items():
    truth = variables[k]['truth']
    reco = variables[k]['reco']
    bounds = variables[k]['range']
    bin_edges = variables[k]['bin_edges']
    bin_edges_1d = variables[k]['bin_edges_1d']
    
    truth_hist, truth_edges = np.histogram(all_sig[truth], bins=bin_edges_1d, weights=all_sig["weights"], range=bounds)
    H, xedges, yedges = np.histogram2d(sel_sig[truth], sel_sig[reco], bins=bin_edges, weights=sel_sig["weights"], range=bounds)
    
    resp = H.T / truth_hist
    
    X, Y = np.meshgrid(xedges,yedges)
    plt.pcolormesh(X, Y, resp, shading='flat') #, norm=LogNorm())
    plt.colorbar()
    plt.xlabel(f'True {k}')
    plt.ylabel(f'Reco {k}')
    
    plt.title(f"Response Matrix")
    label = reco.lstrip("Reco")
    plt.savefig(f'analysis_plots/unfolding_inputs/response_matrix_{data}_{run_combo}_{label}_2bins.pdf', bbox_inches='tight')
    plt.savefig(f'analysis_plots/unfolding_inputs/response_matrix_{data}_{run_combo}_{label}_2bins.png', bbox_inches='tight')
    plt.show()
    plt.clf()
    
    print('Response matrix for', k, ': \n', resp)
    
    # writing these to a file
    resp_str = "{"
    for i in range(resp.shape[0]):
        for j in range(resp.shape[1]):
            resp_str += f"{resp[i][j]},"
    resp_str = resp_str[:-1] # to remove the last comma
    resp_str += "};"
    print(label)
    with open(f'unfolding_inputs_{data}_{run_combo}.txt', 'a') as f:
        f.writelines([f"{label}", " = ", f"{resp_str} \n"])

In [ ]:
# Calculating the metrics

print("Calculating metrics:")
print()

all_mc = pd.concat([df for k, df in rundata.items() if k!='data' or k!='ext'])

all_sig = all_mc['category_1e1p'] == 12
tot_all_sig = np.sum(all_mc.loc[all_sig, 'weights'])
print('Total candidate signal events:', tot_all_sig)

all_predict = all_mc.query(query, engine='python')

is_sig = all_predict['category_1e1p'] == 12
tot_sig = np.sum(all_predict.loc[is_sig, 'weights'])
tot_bkg = np.sum(all_predict.loc[~is_sig, 'weights'])
tot_evt = np.sum(all_predict['weights'])
print('After cuts:')
print('Total signal events:', tot_sig)
print('Total background events:', tot_bkg)
print('Total events:', tot_evt)
print('sig + bkg =', tot_sig+tot_bkg)
print()

efficiency = (tot_sig / tot_all_sig) * 100
purity = (tot_sig / tot_evt) * 100
print('Efficiency:', efficiency)
print('Purity:', purity)
print()

pred_res = all_predict['interaction'] == 1
tot_res = np.sum(all_predict.loc[pred_res, 'weights'])
print('Total resonance events:', tot_res)
pred_qe = all_predict['interaction'] == 0
tot_qe = np.sum(all_predict.loc[pred_qe, 'weights'])
print('Total quasielastic events:', tot_qe)

print('QE/RES =', tot_qe/tot_res)
print('RES/QE =', tot_res/tot_qe)
print('QE/Total =', tot_qe/tot_evt)

with open(f'unfolding_inputs_{data}_{run_combo}.txt', 'a') as f:
    f.write("\nMetrics:\n")
    f.writelines(['\nTotal candidate signal events = ', f"{tot_all_sig} \n"])
    f.writelines(["After cuts: \n"])
    f.writelines(['Total signal events = ', f"{tot_sig} \n"])
    f.writelines(['Total background events = ', f"{tot_bkg} \n"])
    f.writelines(['Total events = ', f"{tot_evt} \n"])
    f.writelines(['\nPurity = ', f"{purity} % \n"])
    f.writelines(['Efficiency = ', f"{efficiency} % \n"])
    f.writelines(['\nQE/RES = ', f"{tot_qe/tot_res} \n"])
    f.writelines(['RES/QE = ', f"{tot_res/tot_qe} \n"])
    f.writelines(['QE/Total = ', f"{tot_qe/tot_evt} \n"])

f.close()